# Embedding

Mit dem Embedding wandeln wir die Chunks in Vektoren um und speichern die Daten im Chunk.

- Model: deepset/gbert-large
- Input: products_chunked.jsonl
- Output: products_embedded.jsonl

In [29]:
import json
import random
import numpy as np

from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('deepset/gbert-large')

No sentence-transformers model found with name deepset/gbert-large. Creating a new one with mean pooling.


## Daten vorbereiten

Die Texte und die Chunks müssen getrennt verarbeitet werden bzw. in zwei Arrays abgelegt werden, die am dann per Index wieder zusammengeführt werden. Da die Eingangsdaten bereits als Chunks strukturiert sind, genügt es die zu vektorisieren Tete zu entnehmen und die Embeddings danach wieder hinzuzufügen.

In [25]:
chunks = []

with open('../data/processed/products_chunked.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        chunks.append(json.loads(line))

# Text only
texts = [chunk['document'] for chunk in chunks]

#print(texts)

## Embedding

Es werden alle Daten übergeben und in 16er-Schritten encodet. Progressbar ist for fun, Normalisieren ist Standard bei ChromaDB, glaube ich. Da wir die Daten nur für die Ähnlichkeitssuche benötigen ist die Länge und die darin enthaltene semantische Bedeutung nicht relevant.

In [26]:
embeddings = model.encode(
    texts,
    batch_size = 16,
    show_progress_bar = True,
    normalize_embeddings = True,
    convert_to_numpy = True
)

Batches: 100%|██████████| 331/331 [30:25<00:00,  5.52s/it] 


## Zusammenfassen

WIr verbinden mit zip() die Chunks der originalen Datei mit den Embeddings vom Model.

In [27]:
for chunk, embedding in zip (chunks, embeddings):
    chunk['embedding'] = embedding.tolist()

with open('../data/processed/products_embedded.jsonl', 'w', encoding='utf-8') as f:
    for chunk in chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

## Evaluieren der Embeddings

In [30]:
# Längenvergleich
assert len(embeddings) == len(chunks)

# Stichproben
sample_idx = random.sample(range(len(embeddings)), int(len(embeddings) * 0.01))
#sample_idx = [0]
for idx in sample_idx:
    text = chunks[idx]['document']
    embd = embeddings[idx]
    norm = np.linalg.norm(embd)
    test = model.encode([text], normalize_embeddings=True)[0]
    similarity = np.dot(embd, test)

    print(f"Index: {idx}")
    print(f"Text: {text[:60]}...")
    print(f"Shape: {embd}, Norm: {norm:.4f}")
    print(f"Similarity: {similarity:.8f}")

assert not np.any(np.isnan(embeddings))
assert not np.any(np.isinf(embeddings))

print(f"Shape: {embeddings.shape}")
print(f"Dtype: {embeddings.dtype}")

Index: 453
Text: Der Kirsch LABO-288 PRO-ACTIVE hat einen Gradient von 7,4 °C...
Shape: [ 0.00744948 -0.00504931 -0.01007205 ...  0.00373016  0.00764065
 -0.02243659], Norm: 1.0000
Similarity: 1.00000000
Index: 1539
Text: Der Kirsch LABO-288 PRO-ACTIVE hat einen Normalverbrauch von...
Shape: [ 0.02372998  0.01198341 -0.00345045 ...  0.01456727 -0.00740333
 -0.01979515], Norm: 1.0000
Similarity: 1.00000000
Index: 216
Text: Der Kirsch BL-720 Blutkonservenkühlschrank ist mit einer PRO...
Shape: [ 0.00020909  0.00011962  0.00046266 ...  0.00197034 -0.00179797
 -0.00979628], Norm: 1.0000
Similarity: 0.99999994
Index: 2034
Text: Der Kirsch LABO-288 PRO-ACTIVE hat eine Temperaturabweichung...
Shape: [ 0.00775406  0.00310258 -0.00571194 ...  0.00752133 -0.00882986
 -0.01261879], Norm: 1.0000
Similarity: 1.00000024
Index: 3640
Text: Der Kirsch LABO-288 PRO-ACTIVE Laborkühlschrank hat einen ei...
Shape: [ 0.01468216  0.00691097  0.00617777 ...  0.00357857 -0.0136908
 -0.00842649], Norm: 1.0000
S